# Audit GSM8K results in Colab
Select **Runtime → Change runtime type → GPU**. This evaluates the original base model and the public adapter with the repository's plain-text prompt, boxed-answer scoring, greedy decoding, and seed-42 validation split (500 examples). Test has 1,319 examples.

The article reports validation 51.20 → 63.60 → 83.60% and test 41.93 → 51.18 → 73.69% for base → SFT → RLVR. The uploaded adapter is named SFT, but has no model card proving its training stage. Measure it before assigning it to a column.

This uses the corrected Hugging Face evaluator. Historical backend, generation limits, dependency versions, and checkpoint identity must match to claim exact reproduction. In particular, 384 here means **total prompt + completion tokens**. Try 768 separately to measure truncation sensitivity; do not silently replace the original setting.


In [ ]:
%pip install -q "transformers>=4.56,<5" "peft==0.18.1" datasets accelerate flask tqdm
%pip install -q --no-deps "git+https://github.com/mariogemoll/llm-posttraining.git@f828981232a53cfb97d6b4ccb315ab1c146983dc"


After installation, select **Runtime → Restart session** before running the remaining cells. PEFT is pinned to 0.18.1, the version recorded in the uploaded adapter. This avoids the newer PEFT requirement for torchao ≥0.16 conflicting with Colab's preinstalled torchao 0.10.0.


In [ ]:
import gc, hashlib, importlib.metadata, json, subprocess
from pathlib import Path
import torch
from huggingface_hub import HfApi, snapshot_download
from llm_posttraining.eval import evaluate
from llm_posttraining.data import load_gsm8k, PROMPT_TEMPLATE

assert torch.cuda.is_available(), "Select a GPU runtime first."
BASE = "Qwen/Qwen2.5-Math-1.5B"
ADAPTER = "mariogemoll/Qwen2.5-Math-1.5B-GSM8K-SFT-LoRA"
ADAPTER_REVISION = "9155b6ef169e0d08eeacbfc4e60394d3b8ebce95"
BASE_REVISION = HfApi().model_info(BASE).sha
base_path = snapshot_download(BASE, revision=BASE_REVISION)
adapter_path = snapshot_download(ADAPTER, revision=ADAPTER_REVISION)
# Ensure both evaluations load the exact same base snapshot.
adapter_config = Path(adapter_path, "adapter_config.json")
local_adapter = Path("audit_adapter")
local_adapter.mkdir(exist_ok=True)
import shutil
for name in ("adapter_config.json", "adapter_model.safetensors"):
    shutil.copy2(Path(adapter_path, name), local_adapter / name)
cfg = json.loads(adapter_config.read_text())
cfg["base_model_name_or_path"] = base_path
(local_adapter / "adapter_config.json").write_text(json.dumps(cfg, indent=2))
DTYPE = "bfloat16" if torch.cuda.is_bf16_supported(including_emulation=False) else "float16"
print(torch.cuda.get_device_name(), DTYPE)


In [ ]:
# Use 20 for a smoke test, then None for the complete audit.
MAX_EXAMPLES = None
TOTAL_TOKEN_LIMITS = [384]  # Add 768 for a separately labeled sensitivity run.
BATCH_SIZE = 16  # Reduce if memory is insufficient.
out = Path("gsm8k_audit")
out.mkdir(exist_ok=True)
splits = load_gsm8k()
manifest = {
    "code_revision": "f828981232a53cfb97d6b4ccb315ab1c146983dc", "base": BASE, "base_revision": BASE_REVISION,
    "adapter": ADAPTER, "adapter_revision": ADAPTER_REVISION,
    "gpu": torch.cuda.get_device_name(), "dtype": DTYPE, "backend": "hf",
    "batch_size": BATCH_SIZE, "max_examples": MAX_EXAMPLES,
    "total_token_limits": TOTAL_TOKEN_LIMITS, "prompt": PROMPT_TEMPLATE,
    "packages": {p: importlib.metadata.version(p) for p in
                 ["torch", "transformers", "peft", "datasets", "accelerate"]},
    "split_hashes": {name: hashlib.sha256(json.dumps(
        list(splits[name]), sort_keys=True).encode()).hexdigest() for name in ["val", "test"]},
}
(out / "manifest.json").write_text(json.dumps(manifest, indent=2))
(out / "pip-freeze.txt").write_text(subprocess.check_output(
    ["python", "-m", "pip", "freeze"], text=True))
summary = []
for limit in TOTAL_TOKEN_LIMITS:
    for split in ["val", "test"]:
        for label, ckpt in [("base", None), ("uploaded_adapter", str(local_adapter))]:
            acc, results = evaluate(split=split, max_examples=MAX_EXAMPLES,
                ckpt=ckpt, model_id=base_path, batch_size=BATCH_SIZE,
                device="cuda", dtype=DTYPE, max_seq_len=limit, backend="hf")
            row = dict(model=label, split=split, total_token_limit=limit,
                       examples=len(results), correct=sum(r["correct"] for r in results),
                       accuracy_pct=100 * acc)
            summary.append(row)
            (out / f"{label}_{split}_{limit}.json").write_text(json.dumps(results, indent=2))
            (out / "summary.json").write_text(json.dumps(summary, indent=2))
            gc.collect()
            torch.cuda.empty_cache()
print(json.dumps(summary, indent=2))


In [ ]:
# Download the audit, including every answer and the environment manifest.
from google.colab import files  # ty: ignore[unresolved-import]
shutil.make_archive("gsm8k_audit", "zip", out)
files.download("gsm8k_audit.zip")
